# 🚗 Automotive Object Detection — train it on Google Colab

Run the cells **from top to bottom** (click a cell, press `Shift + Enter`).
You don't need to change any code except your GitHub username in step 2.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`

## 1. Check the GPU is on

In [ ]:
!nvidia-smi

## 2. Get the project code
Replace `souadessemyhij1999` with your GitHub username.

In [ ]:
GITHUB_USER = "souadessemyhij1999"
!git clone https://github.com/{GITHUB_USER}/automotive-object-detection.git
%cd automotive-object-detection

## 3. Install the libraries (≈1 minute)

In [ ]:
!pip install -q -r requirements.txt

## 4. Baseline: what does a pretrained model already see?
This uses YOLO26 trained on the general COCO dataset — before any fine-tuning on driving data.

In [ ]:
!mkdir -p samples
!wget -q -O samples/street.jpg https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg
!python src/detect.py --source samples/street.jpg --out results/baseline

from IPython.display import Image, display
display(Image("results/baseline/street.jpg", width=600))

## 5. Fine-tune on KITTI (German driving scenes) — ≈45–75 minutes
The KITTI dataset downloads automatically. Keep this tab open while it trains.
If Colab disconnects, re-run steps 2–3 and then this cell.

In [ ]:
!python src/train.py --epochs 50 --batch 16 --device 0

## 6. Evaluate: accuracy per class

In [ ]:
!python src/evaluate.py --device 0
display(Image("results/metrics_per_class.png", width=700))
print(open("results/summary.md").read())

## 7. Training curves and confusion matrix (great for the README)

In [ ]:
display(Image("runs/detect/kitti_yolo26n/results.png", width=900))
display(Image("runs/detect/kitti_yolo26n/confusion_matrix_normalized.png", width=600))

## 8. Try it on your own dashcam photo or video
Upload a file with the 📁 folder icon on the left (into `automotive-object-detection/samples/`), then change the file name below.

In [ ]:
SOURCE = "samples/street.jpg"   # e.g. "samples/my_drive.mp4"
!python src/detect.py --source {SOURCE} --out results/detections
!python src/summarize_detections.py --csv results/detections/detections.csv
display(Image("results/detections/detection_summary.png", width=800))

## 9. Download everything you need for GitHub
This creates `portfolio_results.zip` — unzip it into your local repo folder, then commit and push (on GitHub: **Add file → Upload files**).

In [ ]:
!mkdir -p assets
!cp runs/detect/kitti_yolo26n/results.png assets/training_curves.png
!cp runs/detect/kitti_yolo26n/confusion_matrix_normalized.png assets/confusion_matrix.png
!cp results/metrics_per_class.png assets/metrics_per_class.png
!cp results/detections/*.jpg assets/ 2>/dev/null; true
!zip -qr portfolio_results.zip weights/best.pt results assets
from google.colab import files
files.download("portfolio_results.zip")